In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
import os
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import sacrebleu
import time
import wandb
import torch.utils.checkpoint as cp

### WANDB

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

### CONFIG

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

d_model = 256
num_heads = 4
num_encoder_layers = 4
num_decoder_layers = 4
dropout = 0.1

batch_size = 8         # giảm để tránh OOM
accum_steps = 4        # => effective batch = 32
epochs = 20
learning_rate = 1e-4
patience = 5
max_dec_len = 80
max_seq_len = 128      # truncate input để tránh OOM
use_amp = True
use_checkpointing = True

num_workers = 0
pin_memory = True

Using device: cuda


### TOKENIZER

In [ ]:
class SimpleTokenizer:
    PAD = "<pad>"
    BOS = "<s>"
    EOS = "</s>"
    UNK = "<unk>"

    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.word2id = {}
        self.id2word = {}
        self.fitted = False

    def fit(self, texts):
        freq = {}
        for s in texts:
            for w in s.strip().split():
                freq[w] = freq.get(w, 0) + 1

        vocab = [self.PAD, self.BOS, self.EOS, self.UNK]
        for w, c in sorted(freq.items(), key=lambda x: -x[1]):
            if c >= self.min_freq and w not in vocab:
                vocab.append(w)

        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}
        self.fitted = True

    def encode(self, text, max_length=None):
        toks = text.strip().split()
        ids = [self.word2id[self.BOS]]

        for t in toks:
            ids.append(self.word2id.get(t, self.word2id[self.UNK]))
            if max_length and len(ids) >= max_length - 1:
                break

        ids.append(self.word2id[self.EOS])
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            if isinstance(i, torch.Tensor):
                i = int(i.item())
            if i not in self.id2word:
                continue
            w = self.id2word[i]
            if w not in (self.PAD, self.BOS, self.EOS):
                words.append(w)
        return " ".join(words)

    def vocab_size_(self):
        return len(self.word2id)

In [ ]:
class NMTDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, src_tok, tgt_tok, max_length):
        self.src = src_texts
        self.tgt = tgt_texts
        self.src_tok = src_tok
        self.tgt_tok = tgt_tok
        self.max_length = max_length

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        s = torch.LongTensor(self.src_tok.encode(self.src[idx], max_length=self.max_length))
        t = torch.LongTensor(self.tgt_tok.encode(self.tgt[idx], max_length=self.max_length))
        return s, t


def collate_batch(batch):
    srcs = [item[0] for item in batch]
    tgts = [item[1] for item in batch]

    src_pad = src_tok.word2id[src_tok.PAD]
    tgt_pad = tgt_tok.word2id[tgt_tok.PAD]

    srcs = pad_sequence(srcs, batch_first=True, padding_value=src_pad)
    tgts = pad_sequence(tgts, batch_first=True, padding_value=tgt_pad)

    src_mask = (srcs != src_pad)
    tgt_mask = (tgts != tgt_pad)

    return srcs, tgts, src_mask, tgt_mask

### MODEL

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)].unsqueeze(0).to(x.device)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0,
        self.h = num_heads
        self.d = d_model // num_heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, Lq, D = q.size()
        _, Lk, _ = k.size()

        q = self.q(q).view(B, Lq, self.h, self.d).transpose(1, 2)
        k = self.k(k).view(B, Lk, self.h, self.d).transpose(1, 2)
        v = self.v(v).view(B, Lk, self.h, self.d).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d)

        if mask is not None:
            mask = mask.to(scores.device)
            if mask.dim() == 2:  # [B, Lk] -> [B,1,1,Lk]
                mask = mask.unsqueeze(1).unsqueeze(2)
            mask = mask.bool()

            neg_inf = torch.finfo(scores.dtype).min / 2
            scores = scores.masked_fill(~mask, neg_inf)

        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, Lq, D)

        return self.out(out)


class ReTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, include_ff=True):
        super().__init__()
        self.self1 = MultiHeadAttention(d_model, num_heads)
        self.self2 = MultiHeadAttention(d_model, num_heads)
        self.include_ff = include_ff

        if include_ff:
            self.ff = nn.Sequential(
                nn.Linear(d_model, 4 * d_model),
                nn.ReLU(),
                nn.Linear(4 * d_model, d_model)
            )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model) if include_ff else None
        self.drop = nn.Dropout(0.1)

    def forward(self, x, i, mask):
        a1 = self.self1(x, x, x, mask)
        x = self.norm1(x + self.drop(a1))

        a2 = self.self2(x, x, x, mask)
        x = self.norm2(x + self.drop(a2))

        if self.include_ff:
            ff = self.ff(x)
            x = self.norm3(x + self.drop(ff))

        return x


class ReTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_attn = MultiHeadAttention(d_model, num_heads)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(0.1)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        a = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.drop(a))

        b = self.enc_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.drop(b))
        return x


class ReTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, heads, n_enc, n_dec, use_checkpoint=False):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pos = PositionalEncoding(d_model)

        self.encoder = nn.ModuleList([
            ReTransformerEncoderLayer(d_model, heads, include_ff=(i % 2 == 0))
            for i in range(n_enc)
        ])

        self.decoder = nn.ModuleList([
            ReTransformerDecoderLayer(d_model, heads)
            for _ in range(n_dec)
        ])

        self.out = nn.Linear(d_model, tgt_vocab)
        self.use_checkpointing = use_checkpoint

    def forward(self, src, tgt, src_mask, tgt_mask):
        src = self.pos(self.src_emb(src))
        tgt = self.pos(self.tgt_emb(tgt))

        for i, layer in enumerate(self.encoder):
            if self.use_checkpointing and self.training:
                src = cp.checkpoint(layer, src, i, src_mask, use_reentrant=False)
            else:
                src = layer(src, i, src_mask)

        for layer in self.decoder:
            if self.use_checkpointing and self.training:
                tgt = cp.checkpoint(layer, tgt, src, src_mask, tgt_mask, use_reentrant=False)
            else:
                tgt = layer(tgt, src, src_mask, tgt_mask)

        return self.out(tgt)


### EVALUATION

In [ ]:

def make_tgt_mask(L):
    mask = torch.triu(torch.ones(L, L), diagonal=1) == 0
    return mask.unsqueeze(0).unsqueeze(0).to(device)

def calculate_ppl(loss):
    try:
        return math.exp(loss)
    except OverflowError:
        return float("inf")

@torch.no_grad()
def greedy_decode(model, s_ids, src_tok, tgt_tok, max_len=max_dec_len):
    model.eval()
    src = s_ids.unsqueeze(0).to(device)
    src_mask = (src != src_tok.word2id[src_tok.PAD])

    tgt_ids = [tgt_tok.word2id[tgt_tok.BOS]]

    for _ in range(max_len):
        tgt_tensor = torch.LongTensor(tgt_ids).unsqueeze(0).to(device)
        tgt_mask = make_tgt_mask(len(tgt_ids))

        out = model(src, tgt_tensor, src_mask, tgt_mask)
        next_token = int(out[0, -1].argmax())
        tgt_ids.append(next_token)

        if next_token == tgt_tok.word2id[tgt_tok.EOS]:
            break

    return tgt_ids[1:]


@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    model.eval()
    total = 0
    count = 0

    for src, tgt, src_mask, tgt_mask in loader:
        src = src.to(device)
        tgt = tgt.to(device)
        src_mask = src_mask.to(device)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        tgt_mask = make_tgt_mask(tgt_in.size(1))

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(src, tgt_in, src_mask, tgt_mask)
            loss = criterion(out.reshape(-1, out.size(-1)),
                             tgt_out.reshape(-1))

        total += loss.item()
        count += 1

    return total / count


@torch.no_grad()
def evaluate_bleu(model, dataset):
    hyps, refs = [], []

    for i in range(min(200, len(dataset))):
        s_text = dataset.src[i]
        t_text = dataset.tgt[i]

        s_ids = torch.LongTensor(src_tok.encode(s_text, max_length=max_seq_len))

        gen_ids = greedy_decode(model, s_ids, src_tok, tgt_tok)
        hyp = tgt_tok.decode(gen_ids)

        refs.append([t_text])
        hyps.append(hyp)

    return sacrebleu.corpus_bleu(hyps, refs).score

### LOAD DATA

In [ ]:

def load_iwslt(folder):
    train_en = open(os.path.join(folder, "train.en")).read().splitlines()
    train_vi = open(os.path.join(folder, "train.vi")).read().splitlines()

    dev_en   = open(os.path.join(folder, "tst2012.en")).read().splitlines()
    dev_vi   = open(os.path.join(folder, "tst2012.vi")).read().splitlines()

    test_en  = open(os.path.join(folder, "tst2013.en")).read().splitlines()
    test_vi  = open(os.path.join(folder, "tst2013.vi")).read().splitlines()

    return (train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi)

In [ ]:
def train_pipeline(train_en, train_vi, dev_en, dev_vi, test_en, test_vi):

    global src_tok, tgt_tok

    src_tok = SimpleTokenizer()
    tgt_tok = SimpleTokenizer()
    src_tok.fit(train_en + dev_en + test_en)
    tgt_tok.fit(train_vi + dev_vi + test_vi)

    train_ds = NMTDataset(train_en, train_vi, src_tok, tgt_tok, max_seq_len)
    dev_ds   = NMTDataset(dev_en, dev_vi, src_tok, tgt_tok, max_seq_len)
    test_ds  = NMTDataset(test_en, test_vi, src_tok, tgt_tok, max_seq_len)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_batch, pin_memory=pin_memory, num_workers=num_workers
    )

    dev_loader = DataLoader(
        dev_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_batch, pin_memory=pin_memory, num_workers=num_workers
    )

    model = ReTransformer(
        src_tok.vocab_size_(), tgt_tok.vocab_size_(),
        d_model, num_heads, num_encoder_layers, num_decoder_layers,
        use_checkpointing
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_tok.word2id[tgt_tok.PAD])

    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    try:
        run = wandb.init(
            project="Retransformer-IWSLT15",
            config={
                "batch": batch_size,
                "accum": accum_steps,
                "d_model": d_model,
                "heads": num_heads,
                "max_len": max_seq_len
            },
            reinit=True
        )
        wandb_run = run
    except Exception as e:
        print("wandb init failed:", e)
        wandb_run = None

    best_loss = float("inf")
    patience_count = 0

    print("=== TRAINING START ===")

    for ep in range(1, epochs + 1):
        model.train()
        total_train = 0.0
        optimizer.zero_grad()

        for i, (src, tgt, src_mask, tgt_mask) in enumerate(train_loader):
            src, tgt = src.to(device), tgt.to(device)
            src_mask = src_mask.to(device)

            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            tgt_mask = make_tgt_mask(tgt_in.size(1))

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(src, tgt_in, src_mask, tgt_mask)
                loss = criterion(out.reshape(-1, out.size(-1)), tgt_out.reshape(-1))
                loss = loss / accum_steps

            scaler.scale(loss).backward()

            if (i + 1) % accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_train += (loss.item() * accum_steps)

        train_loss = total_train / max(1, len(train_loader))
        dev_loss   = evaluate_loss(model, dev_loader, criterion)
        dev_ppl    = calculate_ppl(dev_loss)

        print(f"[Epoch {ep}] Train={train_loss:.4f}, Dev={dev_loss:.4f}, PPL={dev_ppl:.2f}")

        if wandb_run:
            wandb_run.log({
                "train/loss": train_loss,
                "dev/loss": dev_loss,
                "dev/ppl": dev_ppl
            })

        # Early stopping
        if dev_loss < best_loss:
            best_loss = dev_loss
            patience_count = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_count += 1
            if patience_count >= patience:
                print("Early stopping!")
                break

    model.load_state_dict(torch.load("best_model.pt", map_location=device))

    print("=== EVALUATING ON TEST ===")
    test_loader = DataLoader(
        test_ds, batch_size=batch_size,
        collate_fn=collate_batch, pin_memory=pin_memory, num_workers=num_workers
    )

    test_loss = evaluate_loss(model, test_loader, criterion)
    test_ppl  = calculate_ppl(test_loss)
    test_bleu = evaluate_bleu(model, test_ds)

    print(f"Test Loss={test_loss:.4f}, PPL={test_ppl:.2f}, BLEU={test_bleu:.2f}")

    if wandb_run:
        wandb_run.log({
            "test/loss": test_loss,
            "test/ppl": test_ppl,
            "test/bleu": test_bleu
        })
        wandb_run.finish()

    return model, test_bleu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
folder = "/kaggle/input/test-transformer/Assignment_nlp/data"
(train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi) = load_iwslt(folder)

model, bleu = train_pipeline(train_en, train_vi, dev_en, dev_vi, test_en, test_vi)

print("Training completed. Final BLEU:", bleu)

/tmp/ipykernel_19/2910015430.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251130_041736-6qp5i04p
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vague-rain-3
wandb: ⭐️ View project at https://wandb.ai/vuminhson-vietnam-national-university-hanoi/Retransformer-IWSLT15
wandb: 🚀 View run at https://wandb.ai/vuminhson-vietnam-national-university-hanoi/Retransformer-IWSLT15/runs/6qp5i04p


=== TRAINING START ===


/tmp/ipykernel_19/2910015430.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_19/1212351374.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[Epoch 1] Train=4.9207, Dev=4.1258, PPL=61.92
[Epoch 2] Train=3.8489, Dev=3.6973, PPL=40.34
[Epoch 3] Train=3.4520, Dev=3.4460, PPL=31.38
[Epoch 4] Train=3.1775, Dev=3.2790, PPL=26.55
[Epoch 5] Train=2.9609, Dev=3.1467, PPL=23.26
[Epoch 6] Train=2.7890, Dev=3.0645, PPL=21.42
[Epoch 7] Train=2.6484, Dev=2.9932, PPL=19.95
[Epoch 8] Train=2.5296, Dev=2.9438, PPL=18.99
[Epoch 9] Train=2.4264, Dev=2.9057, PPL=18.28
[Epoch 10] Train=2.3341, Dev=2.8787, PPL=17.79
[Epoch 11] Train=2.2534, Dev=2.8565, PPL=17.40
[Epoch 12] Train=2.1835, Dev=2.8308, PPL=16.96
[Epoch 13] Train=2.1178, Dev=2.8188, PPL=16.76
[Epoch 14] Train=2.0586, Dev=2.8130, PPL=16.66
[Epoch 15] Train=2.0055, Dev=2.8119, PPL=16.64
[Epoch 16] Train=1.9572, Dev=2.8096, PPL=16.60
[Epoch 17] Train=1.9118, Dev=2.8257, PPL=16.87
[Epoch 18] Train=1.8695, Dev=2.8300, PPL=16.94
[Epoch 19] Train=1.8317, Dev=2.8256, PPL=16.87
[Epoch 20] Train=1.7969, Dev=2.8376, PPL=17.07
=== EVALUATING ON TEST ===
Test Loss=2.6961, PPL=14.82, BLEU=29.26


wandb:                                                                                
wandb: 
wandb: Run history:
wandb:   dev/loss █▆▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
wandb:    dev/ppl █▅▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:  test/bleu ▁
wandb:  test/loss ▁
wandb:   test/ppl ▁
wandb: train/loss █▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:   dev/loss 2.83756
wandb:    dev/ppl 17.07414
wandb:  test/bleu 29.26395
wandb:  test/loss 2.69608
wandb:   test/ppl 14.82155
wandb: train/loss 1.79687
wandb: 
wandb: 🚀 View run vague-rain-3 at: https://wandb.ai/vuminhson-vietnam-national-university-hanoi/Retransformer-IWSLT15/runs/6qp5i04p
wandb: ⭐️ View project at: https://wandb.ai/vuminhson-vietnam-national-university-hanoi/Retransformer-IWSLT15
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20251130_041736-6qp5i04p/logs


Training completed. Final BLEU: 29.263946665839534


In [ ]:
folder = "/content/drive/MyDrive/Assignment_nlp/data"
(train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi) = load_iwslt(folder)


src_tok = SimpleTokenizer()
tgt_tok = SimpleTokenizer()

src_tok.fit(train_en + dev_en + test_en)
tgt_tok.fit(train_vi + dev_vi + test_vi)

print("SRC vocab:", src_tok.vocab_size_())
print("TGT vocab:", tgt_tok.vocab_size_())


SRC vocab: 54646
TGT vocab: 25885


In [ ]:
model = ReTransformer(
    src_tok.vocab_size_(),
    tgt_tok.vocab_size_(),
    d_model=d_model,
    heads=num_heads,
    n_enc=num_encoder_layers,
    n_dec=num_decoder_layers,
    use_checkpoint=False
).to(device)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/Assignment_nlp/best_model.pt", map_location=device)
)

model.eval()
print("✅ Model loaded successfully")


✅ Model loaded successfully


### CHECK PPL

In [ ]:
test_ds = NMTDataset(test_en, test_vi, src_tok, tgt_tok, max_seq_len)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch,
    pin_memory=True
)

criterion = nn.CrossEntropyLoss(
    ignore_index=tgt_tok.word2id[tgt_tok.PAD]
)

test_loss = evaluate_loss(model, test_loader, criterion)
test_ppl  = calculate_ppl(test_loss)

print(f" Test Loss = {test_loss:.4f}")
print(f" Test PPL  = {test_ppl:.2f}")


/tmp/ipython-input-1212351374.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


📉 Test Loss = 2.6961
🔥 Test PPL  = 14.82


In [ ]:
@torch.no_grad()
def evaluate_bleu_standard(model, src_texts, tgt_texts_RAW, max_samples=500):
    model.eval()
    hyps, refs = [], []

    n = min(len(src_texts), max_samples)

    for i in range(n):
        s_text = src_texts[i]
        ref = tgt_texts_RAW[i]
        s_ids = torch.LongTensor(
            src_tok.encode(s_text, max_length=max_seq_len)
        )

        gen_ids = greedy_decode(model, s_ids, src_tok, tgt_tok)
        hyp = tgt_tok.decode(gen_ids)

        hyps.append(hyp)
        refs.append([ref])

    bleu = sacrebleu.corpus_bleu(
        hyps,
        refs,
        tokenize="13a"
    )
    return bleu.score


### BLEU

In [ ]:
test_bleu = evaluate_bleu_standard(
    model,
    test_en,
    test_vi,
    max_samples=500
)

print(f" Test BLEU (standard, greedy) = {test_bleu:.2f}")


🧪 Test BLEU (standard, greedy) = 34.51


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable
total, trainable = count_parameters(model)

print(f"""
Model parameters
----------------
Total params      : {total:,}
Trainable params  : {trainable:,}
Total (Million)   : {total/1e6:.2f} M
""")




Model parameters
----------------
Total params      : 32,539,421
Trainable params  : 32,539,421
Total (Million)   : 32.54 M

